In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv("Real_Estate_Sales_10012020_to_Current.csv")

In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6898 entries, 0 to 7409
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   xrBuildingTypeID         6898 non-null   float64
 1   ParcelID                 6898 non-null   object 
 2   LocationStartNumber      6898 non-null   int64  
 3   ApartmentUnitNumber      6806 non-null   object 
 4   StreetNameAndWay         6898 non-null   object 
 5   xrPrimaryNeighborhoodID  6898 non-null   int64  
 6   LandSF                   6898 non-null   float64
 7   TotalFinishedArea        6898 non-null   float64
 8   LivingUnits              6898 non-null   float64
 9   OwnerLastName            6898 non-null   object 
 10  OwnerFirstName           6898 non-null   object 
 11  PrimaryGrantor           6898 non-null   object 
 12  SaleDate                 6898 non-null   object 
 13  SalePrice                6898 non-null   float64
 14  TotalAppraisedValue      6898

In [41]:
df.isnull().sum()

xrBuildingTypeID            0
ParcelID                    0
LocationStartNumber         0
ApartmentUnitNumber        92
StreetNameAndWay            0
xrPrimaryNeighborhoodID     0
LandSF                      0
TotalFinishedArea           0
LivingUnits                 0
OwnerLastName               0
OwnerFirstName              0
PrimaryGrantor              0
SaleDate                    0
SalePrice                   0
TotalAppraisedValue         0
LegalReference              0
xrSalesValidityID           0
xrDeedID                    0
AssrLandUse                 0
apartment/house             0
dtype: int64

In [42]:
df.nunique()

xrBuildingTypeID             20
ParcelID                   4835
LocationStartNumber         644
ApartmentUnitNumber         678
StreetNameAndWay            419
xrPrimaryNeighborhoodID     159
LandSF                     1317
TotalFinishedArea          2726
LivingUnits                   7
OwnerLastName              3037
OwnerFirstName             1822
PrimaryGrantor             5179
SaleDate                    973
SalePrice                   910
TotalAppraisedValue        2249
LegalReference             6060
xrSalesValidityID            19
xrDeedID                     18
AssrLandUse                   7
apartment/house               2
dtype: int64

In [ ]:
# 2 העמודות אותו דבר
df.groupby('ParcelID')['PropertyID'].nunique().sort_values(ascending=False).head(10)

ParcelID
104-001-013    1
104-001-015    1
105-001-014    1
105-177-001    1
105-177-006    1
105-177-009    1
105-178-019    1
105-178-020    1
105-178-024    1
106-178-001    1
Name: PropertyID, dtype: int64

In [ ]:
# זורק עמודות ייחודיות ופרופרטי שהוא כמו פרסל
df.drop(columns=['OBJECTID','GlobalID','PropertyID'], inplace=True)

KeyError: "['OBJECTID', 'GlobalID', 'PropertyID'] not found in axis"

In [ ]:
# בדיקה ש2 העמודות אותו דבר
df.groupby('AssrLandUse')['xrCompositeLandUseID'].describe()

KeyError: 'Column not found: xrCompositeLandUseID'

In [37]:
df.drop(columns=['xrCompositeLandUseID'], inplace=True)

In [ ]:
# השלמה של OwnerFirstName לפי OwnerLastName 
df = df.copy()

mask_missing = df['OwnerFirstName'].isna()

# LLC → company
df.loc[
    mask_missing & df['OwnerLastName'].str.contains('llc', case=False, na=False),
    'OwnerFirstName'
] = 'company'

# לא LLC → unknown
df.loc[
    mask_missing & ~df['OwnerLastName'].str.contains('llc', case=False, na=False),
    'OwnerFirstName'
] = 'unknown'

# בדיקה
print("OwnerFirstName – השלמה לפי LLC")
print("--------------------------------")
print(f"הושלמו כ-company: {(df['OwnerFirstName'] == 'company').sum()}")
print(f"הושלמו כ-unknown: {(df['OwnerFirstName'] == 'unknown').sum()}")
print(f"נשארו חסרים: {df['OwnerFirstName'].isna().sum()}")


OwnerFirstName – השלמה לפי LLC
--------------------------------
הושלמו כ-company: 1737
הושלמו כ-unknown: 841
נשארו חסרים: 0


In [9]:
#מחיקה של חניון
df= df[df['AssrLandUse']!="CONDO GARAGE" ]

In [10]:
#מחיקת שורות בודדות שחסרות
df= df[~df['PrimaryGrantor'].isnull() & ~df['LegalReference'].isnull() & ~df['xrBuildingTypeID'].isnull()]

In [43]:
# mark as apartment if AssrLandUse contains any of these substrings
pattern = '|'.join(['CONDOMINIMUM', 'APT FOUR', 'MULTI DWLG', 'APT CRDA'])
df['apartment/house'] = np.where(df['AssrLandUse'].str.contains(pattern, case=False, na=False), 1, 2)

In [ ]:
df = df.copy()

# בית → ApartmentUnitNumber = -2 (כולל ערכים לא חסרים)
df.loc[
    df['apartment/house'] == 'house',
    'ApartmentUnitNumber'
] = -2

# דירה → LandSF = -1 (כולל ערכים לא חסרים)
df.loc[
    df['apartment/house'] == 'apartment',
    'LandSF'
] = -1


In [ ]:
# מחיקת שורות עם SalePrice חסר
df = df[~df['SalePrice'].isnull()]

לא רלוונטי


In [11]:
"Check for invalid SalePrice values:"
invalid_price = df['SalePrice'].isna() | df['SalePrice'].isin(list(range(0, 1001)))
df.groupby('xrSalesValidityID').apply(
    lambda x: invalid_price.loc[x.index].mean()
).sort_values(ascending=False)

C:\Users\BNAIA\AppData\Local\Temp\ipykernel_16192\3378443736.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('xrSalesValidityID').apply(


xrSalesValidityID
4     1.000000
5     0.987761
3     0.888889
9     0.868932
12    0.845324
11    0.777419
0     0.714286
16    0.655738
2     0.647887
7     0.500000
19    0.181818
25    0.179954
17    0.107143
15    0.094203
8     0.068966
26    0.018258
1     0.000827
10    0.000000
27    0.000000
dtype: float64

In [12]:
df['AssrLandUse'].value_counts()

AssrLandUse
ONE FAMILY      2476
CONDOMINIMUM    1766
THREE FAMILY    1617
TWO FAMILY      1285
APT FOUR          80
MULTI DWLG         6
APT CRDA           4
Name: count, dtype: int64

In [13]:
summary = (
    df.groupby('xrSalesValidityID').agg(
        total_rows = ('SalePrice', 'size'),
        null_price = ('SalePrice', lambda x: x.isna().sum()),
        price_0 = ('SalePrice', lambda x: (x == 0).sum()),
        price_1 = ('SalePrice', lambda x: (x == 1).sum()),).sort_index())
summary

,total_rows,null_price,price_0,price_1
xrSalesValidityID,,,,
0,14,2,8,0
1,2419,0,2,0
2,142,1,78,11
3,9,0,5,2
4,49,1,36,8
5,1389,47,911,365
7,2,1,0,0
8,29,1,1,0
9,412,77,215,59


In [38]:
# 1. הגדרת מחיר "אמיתי"
df = df.copy()
df['is_real_price'] = df['SalePrice'].notna() & (df['SalePrice'] > 100)

# 2. פרופיל התנהגות לכל xrSalesValidityID
validity_profile = (
    df
    .groupby('xrSalesValidityID')['is_real_price']
    .mean()
)

# 3. הגדרת סף החלטה (רוב ברור)
REAL_PRICE_THRESHOLD = 0.5

real_price_codes = validity_profile[validity_profile >= REAL_PRICE_THRESHOLD].index
non_real_price_codes = validity_profile[validity_profile < REAL_PRICE_THRESHOLD].index

# 4. שמירת גודל לפני סינון
rows_before = len(df)

# 5. סינון רשומות לא עקביות
mask_consistent = (
    (df['xrSalesValidityID'].isin(real_price_codes) & df['is_real_price']) |
    (df['xrSalesValidityID'].isin(non_real_price_codes) & ~df['is_real_price'])
)

df_clean = df[mask_consistent].copy()

# 6. כמה נמחקו
rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print(f"Rows before cleaning: {rows_before}")
print(f"Rows after cleaning:  {rows_after}")
print(f"Rows removed:        {rows_removed}")


Rows before cleaning: 7234
Rows after cleaning:  6711
Rows removed:        523
